# 🔥 Advanced · Unsloth — fast fine-tuning

> 🔥 **Advanced / heavy lab.** The same QLoRA SFT, ~2x faster and lighter via Unsloth's fused kernels — then export GGUF for llama.cpp / Ollama.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **15–30 min** including downloads. Built on the official **[unslothai/unsloth](https://github.com/unslothai/unsloth)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

A faster path than the plain QLoRA lab when the free T4 is tight.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — up to ~8B 4-bit (≈2× faster) | A100 40/80 GB for bigger models / longer ctx |
| **Storage** | base + adapter; GGUF export ~ 0.3–4 GB | as QLoRA; 50–150 GB |
| **Time** | 60 steps ~ 5–15 min | full dataset ~ a few hours |

**Full pipeline (scale-up):** raise max_steps / full dataset; export merged or GGUF for serving.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q unsloth

## 1 · Load a 4-bit model and attach LoRA (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-0.5B-Instruct", max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

## 2 · Data

In [ ]:
from datasets import load_dataset
ds = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

## 3 · Train (TRL SFTTrainer on the Unsloth model)

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(model=model, train_dataset=ds, processing_class=tok,
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4, max_steps=60,
        learning_rate=2e-4, logging_steps=10, bf16=True, max_seq_length=2048,
        dataset_text_field="text", output_dir="unsloth-out", report_to="none"))
trainer.train()

## 4 · Generate, then save (LoRA · merged · GGUF)

In [ ]:
FastLanguageModel.for_inference(model)
ids = tok.apply_chat_template([{"role": "user", "content": "Explain QLoRA simply."}], add_generation_prompt=True, return_tensors="pt").to(model.device)
print(tok.decode(model.generate(ids, max_new_tokens=160)[0][ids.shape[1]:], skip_special_tokens=True))
model.save_pretrained("unsloth-lora"); tok.save_pretrained("unsloth-lora")
# merged 16-bit:  model.save_pretrained_merged("unsloth-merged", tok, save_method="merged_16bit")
# GGUF for llama.cpp / Ollama:  model.save_pretrained_gguf("unsloth-gguf", tok, quantization_method="q4_k_m")

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`.

## How this links to tracks A–D
Same reach as QLoRA, just faster — adapt any course backbone:
- **A · Human** · **B · 3D** · **C · Egocentric** · **D · Scene / world**.

## Notes & next steps
- Unsloth ships official Colab notebooks per model — match its pinned versions if install hiccups.
- Export **GGUF**, then run locally with **llama.cpp** or **Ollama**.
- Score the result with the lm-eval-harness lab.